In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# transformation on images
# image => scale (0,1) => normalize() in range (-1,1)
transform = transforms.Compose([              
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]) 

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

 10%|█         | 17.2M/170M [2:55:14<26:04:47, 1.63kB/s]
  8%|▊         | 14.2M/170M [1:10:30<12:54:43, 3.36kB/s]


ConnectionAbortedError: [WinError 10053] An established connection was aborted by the software in your host machine

In [3]:
# trainset

NameError: name 'trainset' is not defined

In [ ]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

Build the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            #1st layer
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernel=2, size=2, stride=1

            #2nd layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernel=2, size=2, stride=1

            #3rd layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)  # kernel=2, size=2, stride=1
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            #output
            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattering
        x = self.fc_layers(x)

        return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = nn.Adam(model.parameters())

# Training the CNN

In [ ]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels)  # loss fnx
        loss.backward() # BP
        optimizer.step()  # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

In [ ]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels * 100}")